# Quick Learning Curve Sandbox

This notebook keeps the workflow minimal: it reuses `DataPreparation` for preprocessing/poisoning, calls the training utilities from `model.py`, and overlays the validation learning curves for multiple FrogDQ modes (default: `none`, `inertia`, `samplewise`, `dirichlet`, `q`).
Provide the dataset/poisoning parameters in the configuration cell, run the last cell, and you will get an on-the-fly plot plus a short metric summary.
If you plug in a brand-new dataset, make sure it follows the same schema handled in `DataPreparation` (or extend that class with your loader) so the preprocessing and poisoning steps stay compatible.


In [1]:
from typing import Sequence, Tuple, Dict, Optional

import matplotlib.pyplot as plt
import torch

from DataPreparation import DataPreparation
from model import build_model, train, parse_frogdq_mode


def _detect_device() -> torch.device:
    """Pick the best available accelerator for local experiments."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = _detect_device()
print(f"Using device: {DEVICE}")


Using device: mps


In [2]:
def prepare_dataset(
    dataset_name: str,
    *,
    splitting_perc_train_test: float,
    splitting_perc_test_val: float,
    features_percentage: float,
    poisoning_percentage: float,
    random_state: int,
) -> Dict:
    """Helper that wraps DataPreparation so we only need to provide the name."""
    prep = DataPreparation(dataset_name=dataset_name)
    prep.load()
    return prep.run_preprocessing(
        splitting_perc_train_test=splitting_perc_train_test,
        splitting_perc_test_val=splitting_perc_test_val,
        features_percentage=features_percentage,
        poisoning_percentage=poisoning_percentage,
        random_state=random_state,
    )


def plot_learning_curves(mode_histories: Sequence[Tuple[str, dict]], metric: str = "val_bal_acc") -> None:
    """Overlay validation learning curves for each FrogDQ mode."""
    titles = {
        "val_loss": "Validation loss",
        "val_acc": "Validation accuracy",
        "val_bal_acc": "Validation balanced accuracy",
        "val_f1": "Validation F1",
        "val_auc": "Validation AUC",
    }
    ylabel = titles.get(metric, metric.replace("_", " " ).title())

    if not mode_histories:
        print("No histories available for plotting.")
        return

    plt.figure(figsize=(10, 6))
    plotted = False
    markers = ["o", "s", "^", "D", "v", "P", "X", "*", "h", "<", ">"]
    colors = list(plt.cm.tab10.colors) + list(plt.cm.tab20.colors)
    for idx, (mode_name, history) in enumerate(mode_histories):
        values = history.get(metric, [])
        if not values:
            continue
        epochs = range(1, len(values) + 1)
        color = colors[idx % len(colors)]
        marker = markers[idx % len(markers)]
        plt.plot(
            epochs,
            values,
            label=mode_name,
            linewidth=2.0,
            color=color,
            marker=marker,
            markersize=5,
            markerfacecolor="white",
            markeredgecolor=color,
            markeredgewidth=1.0,
        )
        plotted = True

    if not plotted:
        print(f"Metric '{metric}' not found in the available histories.")
        return

    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(f"Validation curve — {ylabel}")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

def run_learning_curve_experiment(
    *,
    dataset_name: str,
    poisoning_key: str,
    frogdq_modes: Sequence[str],
    features_percentage: float = 0.2,
    poisoning_percentage: float = 0.2,
    splitting_perc_train_test: float = 0.8,
    splitting_perc_test_val: float = 0.5,
    random_state: int = 42,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    optimizer: str = "sgd",
    model_arch: str = "mlp",
    metric: str = "val_bal_acc",
    show_training_logs: bool = False,
    train_overrides: Optional[Dict] = None,
) -> Dict[str, dict]:
    """
    Run preprocessing once, train over the requested FrogDQ modes, plot the curves,
    and return the raw histories (keyed by canonical mode name).
    """
    data_bundle = prepare_dataset(
        dataset_name,
        splitting_perc_train_test=splitting_perc_train_test,
        splitting_perc_test_val=splitting_perc_test_val,
        features_percentage=features_percentage,
        poisoning_percentage=poisoning_percentage,
        random_state=random_state,
    )

    available_splits = sorted(
        k for k, v in data_bundle.items() if isinstance(v, dict) and 'X_train' in v
    )
    if poisoning_key not in data_bundle:
        raise ValueError(
            f"Poisoning split '{poisoning_key}' not found. Choose among: {available_splits}"
        )

    split = data_bundle[poisoning_key]
    class_count = int(torch.unique(data_bundle['y_train']).numel())
    mode_histories: list[Tuple[str, dict]] = []
    summaries: list[Tuple[str, Optional[float]]] = []
    extra_train_args = dict(train_overrides or {})

    for requested_mode in frogdq_modes:
        modules = parse_frogdq_mode(requested_mode)
        canonical_name = modules.canonical
        active_modules = set(modules.active)
        requested_modules = set(modules.requested)
        use_gate = "inertia" in active_modules

        q_vec = split['q'] if use_gate else None
        r_vec = split.get('r') if "samplewise" in active_modules else None
        gate_init_vector = split['q'].tolist() if "q" in requested_modules else None

        print(f"Training mode: {canonical_name}")
        model = build_model(
            input_dim=split['X_train'].shape[1],
            output_dim=class_count,
            arch=model_arch,
            use_frogdq=use_gate,
            gate_init_vector=gate_init_vector,
            random_state=random_state,
        )

        history = train(
            model=model,
            X_train=split['X_train'],
            y_train=data_bundle['y_train'],
            X_val=split['X_val'],
            y_val=data_bundle['y_val'],
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            optimizer=optimizer,
            frogdq_mode=canonical_name,
            q_vec=q_vec,
            r_vec=r_vec,
            device=DEVICE,
            verbose=show_training_logs,
            random_state=random_state,
            **extra_train_args,
        )

        mode_histories.append((canonical_name, history))
        metric_values = history.get(metric, [])
        summaries.append((canonical_name, metric_values[-1] if metric_values else None))

    plot_learning_curves(mode_histories, metric=metric)

    print(f"\nFinal {metric} per mode:")
    for mode_name, value in summaries:
        if value is None:
            print(f"  - {mode_name}: n/a")
        else:
            print(f"  - {mode_name}: {value:.4f}")

    return {name: history for name, history in mode_histories}


In [3]:
# ---------------------------------------------------------------------------
# Adjust these dictionaries to point to your dataset and hyper-parameters
# ---------------------------------------------------------------------------
dataset_config = dict(
    dataset_name="miniboone",
    poisoning_key="nan",  # one of: clean, flipping, noise, nan, all
    features_percentage=0.2,
    poisoning_percentage=0.4,
    splitting_perc_train_test=0.6,
    splitting_perc_test_val=0.5,
    random_state=42,
)

training_config = dict(
    epochs=500,
    batch_size=256,
    lr=0.001,
    optimizer="sgd",
    model_arch="mlp",
    metric="val_f1",
    show_training_logs=True,
)

# Extra keyword arguments passed straight to model.train (optional)
train_overrides = dict(
    lambda_prox=0.1,
    frog_temp_tau=2.0,
    frog_temp_eta=1.01,
    lambda_lasso=0.001,
)

frogdq_modes = ["none", "lasso", "dirichlet"]


In [4]:
histories = run_learning_curve_experiment(
    frogdq_modes=frogdq_modes,
    train_overrides=train_overrides,
    **dataset_config,
    **training_config,
)

histories.keys()


/Users/mattia.sabella/PhD - PoliMi/Frog-DQ/Repository/FrogDQ.Federated_Proximal_Gating_with_Data_Quality/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
2025-12-31 17:10:56.403 | INFO     | DataPreparation:run_preprocessing:303 - Row-wise quality correctly computed for all poisoning types.
/Users/mattia.sabella/PhD - PoliMi/Frog-DQ/Repository/FrogDQ.Federated_Proximal_Gating_with_Data_Quality/DataPreparation.py:320: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_dct[type]['q'] = torch.tensor(data_dct[type]['q'], dtype=torch.float32)
/Users/mattia.sabella/PhD - PoliMi/Frog-DQ/Repository/FrogDQ.Federated_Pro

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [11]:
from sklearn.datasets import fetch_openml

ds = fetch_openml(data_id=44128, parser="auto", as_frame=True)
X = ds.data.copy()
y = ds.target.copy().to_numpy()

In [12]:
y

array(['False', 'False', 'False', ..., 'True', 'True', 'True'],
      shape=(72998,), dtype=object)